# 06. Patient-level Inference Pipeline

## Objective

Integrate the frozen Clinical+RNA CoxNet model and patient-level interpretation modules into a single inference pipeline.

For a single patient, the pipeline will provide:

1. PFI risk score
2. Patient-level feature contributions
3. Top-5 Molecular Twins from the training cohort
4. Molecular support assessment

## Validation principle

The reconstructed inference pipeline must reproduce the existing frozen
Clinical+RNA test risk scores before being used for patient-level inference.

No model refitting or test-set-based preprocessing is allowed.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

print("PROJECT ROOT:", PROJECT_ROOT)

check_dirs = [
    "results",
    "data/processed",
    "models",
]

for d in check_dirs:
    path = PROJECT_ROOT / d

    print(f"\n{'=' * 70}")
    print(f"[{d}]")
    print("=" * 70)

    if not path.exists():
        print("Directory does not exist.")
        continue

    files = sorted(
        p.relative_to(PROJECT_ROOT)
        for p in path.rglob("*")
        if p.is_file()
    )

    if not files:
        print("No files.")
    else:
        for f in files:
            print(f)

PROJECT ROOT: C:\kirc-genomic-prognosis

[results]
results\molecular_support_config.csv
results\molecular_twin_features.csv
results\molecular_twin_results.csv
results\molecular_twin_train_reference.parquet
results\ood_results.csv
results\patient_molecular_support_summary.csv

[data/processed]
data\processed\kirc_modeling_cohort.parquet
data\processed\rna_tpm_modeling_raw.parquet
data\processed\splits.csv

[models]
Directory does not exist.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

keywords = [
    "prediction",
    "coefficient",
    "coef",
    "cox",
    "clinical",
    "rna",
    "feature",
    "scaler",
]

print("Potential model/inference artifacts\n")

for p in sorted(PROJECT_ROOT.rglob("*")):
    if not p.is_file():
        continue

    name = p.name.lower()

    if any(k in name for k in keywords):
        try:
            size_kb = p.stat().st_size / 1024
        except OSError:
            size_kb = float("nan")

        print(f"{p.relative_to(PROJECT_ROOT)}  [{size_kb:.1f} KB]")

Potential model/inference artifacts

data\interim\kirc_clinical_pfi.parquet  [11.5 KB]
data\interim\rna_gene_annotation.parquet  [328.9 KB]
data\interim\rna_tpm_matched.parquet  [59259.2 KB]
data\interim\rna_tpm_patient.parquet  [59259.2 KB]
data\manifests\gdc_manifest_kirc_rna.txt  [86.7 KB]
data\processed\rna_tpm_modeling_raw.parquet  [59117.1 KB]
data\raw\rna\000d81dd-9ba4-4852-9090-2bf22f6483f0\09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv  [4135.9 KB]
data\raw\rna\000d81dd-9ba4-4852-9090-2bf22f6483f0\logs\09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv.parcel  [0.4 KB]
data\raw\rna\00174492-1adc-4310-8147-a622befbb466\84e40c7e-2f60-4461-b14c-0859697295a4.rna_seq.augmented_star_gene_counts.tsv  [4155.0 KB]
data\raw\rna\00174492-1adc-4310-8147-a622befbb466\logs\84e40c7e-2f60-4461-b14c-0859697295a4.rna_seq.augmented_star_gene_counts.tsv.parcel  [0.4 KB]
data\raw\rna\00ddbdd1-f29d-438b-820b-aacb111a3ec4\e635dbc7-ef8d-45e2-acd2-

In [3]:
from pathlib import Path

# 프로젝트 루트 찾기
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("PROJECT ROOT:", ROOT)

# 확인할 폴더
folders = [
    "results",
    "models",
    "data/processed",
    "data/interim",
]

keywords = [
    "clinical",
    "rna",
    "coef",
    "prediction",
    "scaler",
    "feature",
    "model",
]

for folder in folders:
    path = ROOT / folder

    print("\n" + "=" * 70)
    print(f"[{folder}]")

    if not path.exists():
        print("폴더 없음")
        continue

    files = [
        f for f in path.rglob("*")
        if f.is_file()
        and any(k in f.name.lower() for k in keywords)
    ]

    if not files:
        print("관련 파일 없음")
    else:
        for f in files:
            print(f.relative_to(ROOT))

PROJECT ROOT: c:\kirc-genomic-prognosis

[results]
results\molecular_twin_features.csv

[models]
폴더 없음

[data/processed]
data\processed\kirc_modeling_cohort.parquet
data\processed\rna_tpm_modeling_raw.parquet

[data/interim]
data\interim\kirc_clinical_pfi.parquet
data\interim\rna_gene_annotation.parquet
data\interim\rna_tpm_matched.parquet
data\interim\rna_tpm_patient.parquet


In [4]:
# results 폴더 전체 파일 확인
results_dir = ROOT / "results"

print("=== results/ 전체 파일 ===")

if results_dir.exists():
    for f in sorted(results_dir.rglob("*")):
        if f.is_file():
            print(f.relative_to(ROOT))
else:
    print("results 폴더 없음")

=== results/ 전체 파일 ===
results\molecular_support_config.csv
results\molecular_twin_features.csv
results\molecular_twin_results.csv
results\molecular_twin_train_reference.parquet
results\ood_results.csv
results\patient_molecular_support_summary.csv


In [5]:
notebook_dir = ROOT / "notebooks"

print("=== notebooks ===")

for f in sorted(notebook_dir.glob("*.ipynb")):
    print(f.name)

=== notebooks ===
00_clinical_data_audit.ipynb
01_rna_data_audit.ipynb
02_split_preprocessing.ipynb
03_results_visualization.ipynb
04_xai_interpretation.ipynb
05_molecular_twin_ood.ipynb
06_patient_inference.ipynb


In [6]:
print("=== Python scripts ===")

py_files = sorted(
    f for f in ROOT.rglob("*.py")
    if ".git" not in f.parts
)

if not py_files:
    print("Python 파일 없음")
else:
    for f in py_files:
        print(f.relative_to(ROOT))

=== Python scripts ===
Python 파일 없음


In [7]:
import json

search_terms = [
    "Coxnet",
    "CoxPH",
    "clinical_rna",
    "coef_",
    "predict",
    "alpha",
    "l1_ratio",
    "StandardScaler",
    "SelectKBest",
    "top_500",
]

for nb_name in [
    "02_split_preprocessing.ipynb",
    "03_results_visualization.ipynb",
    "04_xai_interpretation.ipynb",
]:
    nb_path = ROOT / "notebooks" / nb_name

    print("\n" + "=" * 80)
    print(nb_name)
    print("=" * 80)

    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    found = False

    for i, cell in enumerate(nb["cells"]):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(term.lower() in source.lower() for term in search_terms):
            found = True
            print(f"\n--- CODE CELL {i} ---")
            print(source[:5000])

    if not found:
        print("관련 코드 없음")


02_split_preprocessing.ipynb

--- CODE CELL 26 ---
from sklearn.preprocessing import StandardScaler

# Fold 1 다시 가져오기
fit_idx, cv_idx = next(
    cv.split(train_df, train_df["pfi_event"])
)

fit_df = train_df.iloc[fit_idx].copy()
cv_df_fold = train_df.iloc[cv_idx].copy()

fit_ids = fit_df["patient_id"]
cv_ids_fold = cv_df_fold["patient_id"]

# Fold 1에서 선정된 Top-500 genes
top500_genes = fold_topk[1][500]

# log2(TPM + 1)
X_fit = np.log2(
    rna_tpm.loc[fit_ids, top500_genes] + 1
)

X_cv = np.log2(
    rna_tpm.loc[cv_ids_fold, top500_genes] + 1
)

# Fit에서만 scaling 학습
scaler = StandardScaler()

X_fit_scaled = scaler.fit_transform(X_fit)
X_cv_scaled = scaler.transform(X_cv)

# survival target
y_fit = Surv.from_arrays(
    event=fit_df["pfi_event"].astype(bool).to_numpy(),
    time=fit_df["pfi_time"].astype(float).to_numpy()
)

y_cv = Surv.from_arrays(
    event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
    time=cv_df_fold["pfi_time"].astype(float).to_numpy()
)

print("X Fit:", X_fi

In [9]:
import json

nb_path = ROOT / "notebooks" / "02_split_preprocessing.ipynb"

with open(nb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

cell_num = 26
source = "".join(nb["cells"][cell_num]["source"])

print(source)

from sklearn.preprocessing import StandardScaler

# Fold 1 다시 가져오기
fit_idx, cv_idx = next(
    cv.split(train_df, train_df["pfi_event"])
)

fit_df = train_df.iloc[fit_idx].copy()
cv_df_fold = train_df.iloc[cv_idx].copy()

fit_ids = fit_df["patient_id"]
cv_ids_fold = cv_df_fold["patient_id"]

# Fold 1에서 선정된 Top-500 genes
top500_genes = fold_topk[1][500]

# log2(TPM + 1)
X_fit = np.log2(
    rna_tpm.loc[fit_ids, top500_genes] + 1
)

X_cv = np.log2(
    rna_tpm.loc[cv_ids_fold, top500_genes] + 1
)

# Fit에서만 scaling 학습
scaler = StandardScaler()

X_fit_scaled = scaler.fit_transform(X_fit)
X_cv_scaled = scaler.transform(X_cv)

# survival target
y_fit = Surv.from_arrays(
    event=fit_df["pfi_event"].astype(bool).to_numpy(),
    time=fit_df["pfi_time"].astype(float).to_numpy()
)

y_cv = Surv.from_arrays(
    event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
    time=cv_df_fold["pfi_time"].astype(float).to_numpy()
)

print("X Fit:", X_fit_scaled.shape)
print("X CV :", X_cv_scaled.shape)
p

In [10]:
# Cell 26에서 필요한 줄 주변만 확인
lines = source.splitlines()

keywords = [
    "StandardScaler",
    "scaler",
    "fit_transform",
    "transform",
    "X_fit_scaled",
    "X_cv_scaled",
    "clinical",
    "Coxnet",
    "coef",
    "alpha",
]

for i, line in enumerate(lines):
    if any(k.lower() in line.lower() for k in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)

        print("\n" + "-" * 70)
        print(f"LINES {start + 1}-{end}")

        for j in range(start, end):
            print(f"{j + 1:03d}: {lines[j]}")


----------------------------------------------------------------------
LINES 1-5
001: from sklearn.preprocessing import StandardScaler
002: 
003: # Fold 1 다시 가져오기
004: fit_idx, cv_idx = next(
005:     cv.split(train_df, train_df["pfi_event"])

----------------------------------------------------------------------
LINES 24-31
024: )
025: 
026: # Fit에서만 scaling 학습
027: scaler = StandardScaler()
028: 
029: X_fit_scaled = scaler.fit_transform(X_fit)
030: X_cv_scaled = scaler.transform(X_cv)
031: 

----------------------------------------------------------------------
LINES 26-33
026: # Fit에서만 scaling 학습
027: scaler = StandardScaler()
028: 
029: X_fit_scaled = scaler.fit_transform(X_fit)
030: X_cv_scaled = scaler.transform(X_cv)
031: 
032: # survival target
033: y_fit = Surv.from_arrays(

----------------------------------------------------------------------
LINES 27-34
027: scaler = StandardScaler()
028: 
029: X_fit_scaled = scaler.fit_transform(X_fit)
030: X_cv_scaled = scaler.transform(

In [11]:
# 02 notebook에서 CoxNet / Clinical+RNA 모델 학습 셀 위치 찾기

with open(ROOT / "notebooks" / "02_split_preprocessing.ipynb",
          "r", encoding="utf-8") as f:
    nb02 = json.load(f)

terms = [
    "CoxnetSurvivalAnalysis",
    "clinical_rna",
    "Clinical + RNA",
    "clinical+rna",
    "l1_ratio",
]

for i, cell in enumerate(nb02["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if any(term.lower() in src.lower() for term in terms):
        print(f"CELL {i}")
        print("-" * 60)

        # 너무 길게 출력하지 않고 첫 1200자만
        print(src[:1200])
        print()

CELL 27
------------------------------------------------------------
import numpy
import scipy
import sklearn
import sksurv

from sksurv.linear_model import CoxnetSurvivalAnalysis

print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("scikit-survival:", sksurv.__version__)
print("CoxNet import OK")

CELL 28
------------------------------------------------------------
from sksurv.linear_model import CoxnetSurvivalAnalysis

coxnet_test = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

coxnet_test.fit(X_fit_scaled, y_fit)

print("alpha 개수:", len(coxnet_test.alphas_))
print("가장 큰 alpha:", coxnet_test.alphas_[0])
print("가장 작은 alpha:", coxnet_test.alphas_[-1])
print("coefficient matrix shape:", coxnet_test.coef_.shape)

CELL 30
------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import

In [12]:
# 02 notebook의 핵심 모델링 Cell 28을 구간별로 출력

cell28 = "".join(nb02["cells"][28]["source"])
lines28 = cell28.splitlines()

print("Cell 28 총 줄 수:", len(lines28))

# 우선 앞부분 1~100줄
for i, line in enumerate(lines28[:100], start=1):
    print(f"{i:03d}: {line}")

Cell 28 총 줄 수: 15
001: from sksurv.linear_model import CoxnetSurvivalAnalysis
002: 
003: coxnet_test = CoxnetSurvivalAnalysis(
004:     l1_ratio=0.5,
005:     n_alphas=50,
006:     alpha_min_ratio=0.01,
007:     max_iter=100000
008: )
009: 
010: coxnet_test.fit(X_fit_scaled, y_fit)
011: 
012: print("alpha 개수:", len(coxnet_test.alphas_))
013: print("가장 큰 alpha:", coxnet_test.alphas_[0])
014: print("가장 작은 alpha:", coxnet_test.alphas_[-1])
015: print("coefficient matrix shape:", coxnet_test.coef_.shape)


In [13]:
# 02 notebook에서 "최종 모델 / validation / test" 관련 셀 찾기

terms = [
    "best_alpha",
    "best_model",
    "final_model",
    "test_df",
    "val_df",
    "X_test",
    "X_val",
    "c_index",
    "concordance_index",
]

for i, cell in enumerate(nb02["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if any(term.lower() in src.lower() for term in terms):
        print(f"\nCELL {i}")
        print("-" * 60)
        print(src[:1500])


CELL 3
------------------------------------------------------------
from sklearn.model_selection import train_test_split

SEED = 42

# 1차: Train 70% / 임시셋 30%
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.30,
    random_state=SEED,
    stratify=cohort["pfi_event"]
)

# 2차: 임시셋을 Validation / Test로 절반씩 분할
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["pfi_event"]
)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))
print("Total      :", len(train_df) + len(val_df) + len(test_df))

CELL 4
------------------------------------------------------------
def check_event_ratio(df, name):
    event_count = int(df["pfi_event"].sum())
    event_rate = df["pfi_event"].mean()

    print(
        f"{name:10s} | "
        f"N={len(df):3d} | "
        f"Event={event_count:3d} | "
        f"Event rate={event_rate:.3f}"
    )

check_event_ratio(train_df, "Train"

In [14]:
for i, cell in enumerate(nb02["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", [])).strip()

    if not src:
        continue

    first_lines = src.splitlines()[:3]

    print(f"\nCELL {i}")
    for line in first_lines:
        print("   ", line[:150])


CELL 1
    from pathlib import Path
    
    import numpy as np

CELL 2
    print("환자 수:", len(cohort))
    print("고유 patient_id:", cohort["patient_id"].nunique())
    

CELL 3
    from sklearn.model_selection import train_test_split
    
    SEED = 42

CELL 4
    def check_event_ratio(df, name):
        event_count = int(df["pfi_event"].sum())
        event_rate = df["pfi_event"].mean()

CELL 5
    splits = pd.concat([
        train_df[["patient_id"]].assign(split="train"),
        val_df[["patient_id"]].assign(split="validation"),

CELL 7
    splits_check = pd.read_csv("../data/processed/splits.csv")
    
    print(splits_check["split"].value_counts())

CELL 8
    train_ids = splits_check.loc[
        splits_check["split"] == "train", "patient_id"
    ]

CELL 9
    min_tpm = 1.0
    min_fraction = 0.20
    

CELL 10
    rna_train_filtered = rna_train.loc[:, keep_genes].copy()
    rna_val_filtered = rna_val.loc[:, keep_genes].copy()
    rna_test_filtered = rna_test.loc[:, keep_genes]

In [15]:
for i, cell in enumerate(nb02["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if "clinical" in src.lower() and "rna" in src.lower():
        first_line = next(
            (x.strip() for x in src.splitlines() if x.strip()),
            ""
        )
        print(f"CELL {i}: {first_line[:120]}")

CELL 1: from pathlib import Path
CELL 44: FINAL_CLIN_L1_RATIO = 0.90
CELL 47: # =========================================
CELL 48: # Clinical + RNA : Fold 1 결합 QC
CELL 49: # =========================================
CELL 52: # =========================================
CELL 53: # =========================================
CELL 54: # =========================================
CELL 64: # =========================================
CELL 69: # =========================================
CELL 71: # ============================================================
CELL 73: # ============================================================
CELL 74: # ============================================================
CELL 76: # =========================================
CELL 77: # =========================================
CELL 78: # ============================================================
CELL 80: # ============================================================
CELL 81: # ==========================================

In [16]:
for i in range(47, 55):
    cell = nb02["cells"][i]

    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    lines = [x.strip() for x in src.splitlines() if x.strip()]

    print(f"\n=== CELL {i} ===")

    # 앞 8줄만
    for line in lines[:8]:
        print(line[:180])


=== CELL 47 ===
# =========================================
# RNA-only + Clinical-only 결과 저장
# =========================================
# 1. RNA 전체 Top-1000 coefficient
rna_coef_df = pd.DataFrame({
"gene_id": final_genes,
"coefficient": rna_coxnet_final.coef_[:, FINAL_ALPHA_IDX]
})

=== CELL 48 ===
# Clinical + RNA : Fold 1 결합 QC
fit_idx, cv_idx = next(
cv.split(train_df, train_df["pfi_event"])
)
fit_df = train_df.iloc[fit_idx].copy()
cv_df_fold = train_df.iloc[cv_idx].copy()
# -------------------------
# Clinical

=== CELL 49 ===
# =========================================
# Clinical + RNA CoxNet
# 5-fold CV hyperparameter tuning
# =========================================
COMBINED_K_VALUES = [500, 1000, 2000, 5000]
COMBINED_L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]
combined_cv_results = []
for fold, (fit_idx, cv_idx) in enumerate(

=== CELL 50 ===
combined_cv_summary = (
combined_cv_results
.groupby(
["k", "l1_ratio", "alpha_idx"],
as_index=False
)
.agg(
alpha_ratio=("alpha_ratio", "

In [17]:
cell49 = "".join(nb02["cells"][49]["source"])

for i, line in enumerate(cell49.splitlines(), start=1):
    print(f"{i:03d}: {line}")

001: # =========================================
002: # Clinical + RNA CoxNet
003: # 5-fold CV hyperparameter tuning
004: # =========================================
005: 
006: COMBINED_K_VALUES = [500, 1000, 2000, 5000]
007: COMBINED_L1_RATIOS = [0.1, 0.25, 0.5, 0.75, 0.9]
008: 
009: combined_cv_results = []
010: 
011: for fold, (fit_idx, cv_idx) in enumerate(
012:     cv.split(train_df, train_df["pfi_event"]),
013:     start=1
014: ):
015:     fit_df = train_df.iloc[fit_idx].copy()
016:     cv_df_fold = train_df.iloc[cv_idx].copy()
017: 
018:     # ---------------------------------
019:     # 1. Clinical preprocessing
020:     # ---------------------------------
021:     fold_clin_preprocessor = ColumnTransformer(
022:         transformers=[
023:             ("age", "passthrough", ["age"]),
024:             (
025:                 "cat",
026:                 OneHotEncoder(
027:                     categories=[
028:                         ["MALE", "FEMALE"],
029:                      

In [18]:
lines49 = cell49.splitlines()

for i in range(17, 110):
    print(f"{i+1:03d}: {lines49[i]}")

018:     # ---------------------------------
019:     # 1. Clinical preprocessing
020:     # ---------------------------------
021:     fold_clin_preprocessor = ColumnTransformer(
022:         transformers=[
023:             ("age", "passthrough", ["age"]),
024:             (
025:                 "cat",
026:                 OneHotEncoder(
027:                     categories=[
028:                         ["MALE", "FEMALE"],
029:                         ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
030:                         ["G1", "G2", "G3", "G4", "Unknown"]
031:                     ],
032:                     drop=["MALE", "Stage I", "G2"],
033:                     handle_unknown="ignore",
034:                     sparse_output=False
035:                 ),
036:                 ["gender", "stage", "grade"]
037:             )
038:         ]
039:     )
040: 
041:     X_fit_clin = fold_clin_preprocessor.fit_transform(
042:         fit_df[clinical_features]
043:     )
04

In [19]:
for i in range(39, 105):
    print(f"{i+1:03d}: {lines49[i]}")

040: 
041:     X_fit_clin = fold_clin_preprocessor.fit_transform(
042:         fit_df[clinical_features]
043:     )
044: 
045:     X_cv_clin = fold_clin_preprocessor.transform(
046:         cv_df_fold[clinical_features]
047:     )
048: 
049:     # ---------------------------------
050:     # 2. RNA preprocessing
051:     #    Fold Fit에서만 결정
052:     # ---------------------------------
053:     rna_fit_raw = rna_tpm.loc[fit_df["patient_id"]]
054:     rna_cv_raw = rna_tpm.loc[cv_df_fold["patient_id"]]
055: 
056:     # 저발현 필터
057:     min_samples = int(np.ceil(len(fit_df) * 0.20))
058: 
059:     keep_genes = (
060:         (rna_fit_raw >= 1).sum(axis=0)
061:         >= min_samples
062:     )
063: 
064:     rna_fit_filtered = rna_fit_raw.loc[:, keep_genes]
065:     rna_cv_filtered = rna_cv_raw.loc[:, keep_genes]
066: 
067:     # log2(TPM + 1)
068:     rna_fit_log = np.log2(rna_fit_filtered + 1)
069:     rna_cv_log = np.log2(rna_cv_filtered + 1)
070: 
071:     # MAD
072:     rna_median = rn

In [20]:
for i in range(64, 102):
    print(f"{i+1:03d}: {lines49[i]}")

065:     rna_cv_filtered = rna_cv_raw.loc[:, keep_genes]
066: 
067:     # log2(TPM + 1)
068:     rna_fit_log = np.log2(rna_fit_filtered + 1)
069:     rna_cv_log = np.log2(rna_cv_filtered + 1)
070: 
071:     # MAD
072:     rna_median = rna_fit_log.median(axis=0)
073: 
074:     rna_mad = (
075:         rna_fit_log
076:         .sub(rna_median, axis=1)
077:         .abs()
078:         .median(axis=0)
079:         .sort_values(ascending=False)
080:     )
081: 
082:     # Survival target
083:     y_fit = Surv.from_arrays(
084:         event=fit_df["pfi_event"].astype(bool).to_numpy(),
085:         time=fit_df["pfi_time"].astype(float).to_numpy()
086:     )
087: 
088:     y_cv = Surv.from_arrays(
089:         event=cv_df_fold["pfi_event"].astype(bool).to_numpy(),
090:         time=cv_df_fold["pfi_time"].astype(float).to_numpy()
091:     )
092: 
093:     # ---------------------------------
094:     # 3. RNA Top-K별 결합 모델
095:     # ---------------------------------
096:     for k in COMBINED_K

In [21]:
for i in range(97, 125):
    print(f"{i+1:03d}: {lines49[i]}")

098:         top_genes = rna_mad.head(k).index
099: 
100:         X_fit_rna = rna_fit_log[top_genes].to_numpy()
101:         X_cv_rna = rna_cv_log[top_genes].to_numpy()
102: 
103:         # Clinical + RNA
104:         X_fit_combined = np.hstack([
105:             X_fit_clin,
106:             X_fit_rna
107:         ])
108: 
109:         X_cv_combined = np.hstack([
110:             X_cv_clin,
111:             X_cv_rna
112:         ])
113: 
114:         # Fold Fit 기준 scaling
115:         scaler = StandardScaler()
116: 
117:         X_fit_combined = scaler.fit_transform(
118:             X_fit_combined
119:         )
120: 
121:         X_cv_combined = scaler.transform(
122:             X_cv_combined
123:         )
124: 
125:         # ---------------------------------


In [22]:
for i in range(124, len(lines49)):
    print(f"{i+1:03d}: {lines49[i]}")

125:         # ---------------------------------
126:         # 4. CoxNet tuning
127:         # ---------------------------------
128:         for l1_ratio in COMBINED_L1_RATIOS:
129: 
130:             model = CoxnetSurvivalAnalysis(
131:                 l1_ratio=l1_ratio,
132:                 n_alphas=50,
133:                 alpha_min_ratio=0.01,
134:                 max_iter=100000
135:             )
136: 
137:             model.fit(
138:                 X_fit_combined,
139:                 y_fit
140:             )
141: 
142:             alpha_max = model.alphas_[0]
143: 
144:             for alpha_idx, alpha in enumerate(model.alphas_):
145: 
146:                 risk = model.predict(
147:                     X_cv_combined,
148:                     alpha=alpha
149:                 )
150: 
151:                 c_index = concordance_index_censored(
152:                     y_cv["event"],
153:                     y_cv["time"],
154:                     risk
155:                 )[0]
15

In [23]:
for i in range(50, 55):
    cell = nb02["cells"][i]

    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    lines = [x for x in src.splitlines() if x.strip()]

    print(f"\n=== CELL {i} ===")
    for line in lines[:15]:
        print(line)


=== CELL 50 ===
combined_cv_summary = (
    combined_cv_results
    .groupby(
        ["k", "l1_ratio", "alpha_idx"],
        as_index=False
    )
    .agg(
        alpha_ratio=("alpha_ratio", "mean"),
        mean_c_index=("c_index", "mean"),
        std_c_index=("c_index", "std"),
        mean_nonzero=("n_nonzero", "mean")
    )
)
combined_best = (
    combined_cv_summary

=== CELL 51 ===
best_combined_fold_scores = combined_cv_results[
    (combined_cv_results["k"] == 500) &
    (combined_cv_results["l1_ratio"] == 0.50) &
    (combined_cv_results["alpha_idx"] == 18)
][
    ["fold", "alpha", "alpha_ratio", "c_index", "n_nonzero"]
]
print(best_combined_fold_scores)
print()
print(
    "Mean C-index:",
    best_combined_fold_scores["c_index"].mean()
)
print(
    "Std C-index:",

=== CELL 52 ===
# =========================================
# Final Clinical + RNA CoxNet
# Train 370명 재학습 → Validation 평가
# =========================================
FINAL_COMBINED_K = 500
FINAL_COMBINED_L1 = 

In [24]:
cell52 = "".join(nb02["cells"][52]["source"])

for i, line in enumerate(cell52.splitlines(), start=1):
    print(f"{i:03d}: {line}")

001: # =========================================
002: # Final Clinical + RNA CoxNet
003: # Train 370명 재학습 → Validation 평가
004: # =========================================
005: 
006: FINAL_COMBINED_K = 500
007: FINAL_COMBINED_L1 = 0.50
008: FINAL_COMBINED_ALPHA_IDX = 18
009: 
010: # -----------------------------------------
011: # 1. Clinical: 전체 Train에서 preprocessing
012: # -----------------------------------------
013: final_combined_clin_preprocessor = ColumnTransformer(
014:     transformers=[
015:         ("age", "passthrough", ["age"]),
016:         (
017:             "cat",
018:             OneHotEncoder(
019:                 categories=[
020:                     ["MALE", "FEMALE"],
021:                     ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
022:                     ["G1", "G2", "G3", "G4", "Unknown"]
023:                 ],
024:                 drop=["MALE", "Stage I", "G2"],
025:                 handle_unknown="ignore",
026:                 sparse_outp

In [25]:
lines52 = cell52.splitlines()

for i in range(29, 130):
    print(f"{i+1:03d}: {lines52[i]}")

030:     ]
031: )
032: 
033: X_train_clin_combined = (
034:     final_combined_clin_preprocessor
035:     .fit_transform(train_df[clinical_features])
036: )
037: 
038: X_val_clin_combined = (
039:     final_combined_clin_preprocessor
040:     .transform(val_df[clinical_features])
041: )
042: 
043: 
044: # -----------------------------------------
045: # 2. RNA: 전체 Train에서 다시 feature selection
046: # -----------------------------------------
047: rna_train_raw = rna_tpm.loc[train_df["patient_id"]]
048: rna_val_raw = rna_tpm.loc[val_df["patient_id"]]
049: 
050: # 저발현 필터 - Train에서만 결정
051: min_samples = int(np.ceil(len(train_df) * 0.20))
052: 
053: keep_genes = (
054:     (rna_train_raw >= 1).sum(axis=0)
055:     >= min_samples
056: )
057: 
058: rna_train_filtered = rna_train_raw.loc[:, keep_genes]
059: rna_val_filtered = rna_val_raw.loc[:, keep_genes]
060: 
061: # log2(TPM + 1)
062: rna_train_log = np.log2(rna_train_filtered + 1)
063: rna_val_log = np.log2(rna_val_filtered + 1)
064: 
065

In [26]:
for i, line in enumerate(lines52, start=1):
    if 45 <= i <= 140 and any(k in line for k in [
        "mad", "MAD",
        "top", "gene",
        "log2",
        "hstack",
        "StandardScaler",
        "scaler",
        "Coxnet",
        "alpha",
        "fit("
    ]):
        print(f"{i:03d}: {line}")

053: keep_genes = (
058: rna_train_filtered = rna_train_raw.loc[:, keep_genes]
059: rna_val_filtered = rna_val_raw.loc[:, keep_genes]
061: # log2(TPM + 1)
062: rna_train_log = np.log2(rna_train_filtered + 1)
063: rna_val_log = np.log2(rna_val_filtered + 1)
065: # Train MAD
068: rna_train_mad = (
077: final_combined_genes = (
078:     rna_train_mad
084:     rna_train_log[final_combined_genes].to_numpy()
088:     rna_val_log[final_combined_genes].to_numpy()
095: X_train_combined_raw = np.hstack([
100: X_val_combined_raw = np.hstack([
106: final_combined_scaler = StandardScaler()
108: X_train_combined = final_combined_scaler.fit_transform(
112: X_val_combined = final_combined_scaler.transform(
120: combined_coxnet_final = CoxnetSurvivalAnalysis(
122:     n_alphas=50,
123:     alpha_min_ratio=0.01,
127: combined_coxnet_final.fit(
132: combined_final_alpha = (
133:     combined_coxnet_final.alphas_[


In [27]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data"
results_dir = project_root / "results"

print("PROJECT ROOT:", project_root)

ModuleNotFoundError: No module named 'sksurv'

In [28]:
import sys
print(sys.executable)

c:\Users\lovel\miniconda3\envs\kirc\python.exe


In [29]:
import importlib.util

print("sksurv:", importlib.util.find_spec("sksurv"))
print("sklearn:", importlib.util.find_spec("sklearn"))
print("numpy:", importlib.util.find_spec("numpy"))

sksurv: None
sklearn: ModuleSpec(name='sklearn', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001EB30057B90>, origin='c:\\Users\\lovel\\miniconda3\\envs\\kirc\\Lib\\site-packages\\sklearn\\__init__.py', submodule_search_locations=['c:\\Users\\lovel\\miniconda3\\envs\\kirc\\Lib\\site-packages\\sklearn'])
numpy: ModuleSpec(name='numpy', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001EB0FB9ED10>, origin='c:\\Users\\lovel\\miniconda3\\envs\\kirc\\Lib\\site-packages\\numpy\\__init__.py', submodule_search_locations=['c:\\Users\\lovel\\miniconda3\\envs\\kirc\\Lib\\site-packages\\numpy'])


In [30]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data"
results_dir = project_root / "results"

print("PROJECT ROOT:", project_root)

PROJECT ROOT: c:\kirc-genomic-prognosis


In [31]:
# ------------------------------------------------------------
# Load modeling data
# ------------------------------------------------------------

cohort = pd.read_parquet(
    data_dir / "processed" / "kirc_modeling_cohort.parquet"
)

rna_tpm = pd.read_parquet(
    data_dir / "processed" / "rna_tpm_modeling_raw.parquet"
)

print("Cohort shape:", cohort.shape)
print("RNA shape   :", rna_tpm.shape)

print("\nCohort columns:")
print(cohort.columns.tolist())

print("\nRNA index sample:")
print(rna_tpm.index[:3].tolist())

Cohort shape: (529, 10)
RNA shape   : (529, 19962)

Cohort columns:
['patient_id', 'age', 'gender', 'stage', 'grade', 'pfi_event', 'pfi_time', 'sample_id', 'file_id', 'file_name']

RNA index sample:
['TCGA-3Z-A93Z', 'TCGA-6D-AA2E', 'TCGA-A3-3306']


In [32]:
from sklearn.model_selection import train_test_split

SEED = 42

# 70% Train / 30% temporary
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.30,
    random_state=SEED,
    stratify=cohort["pfi_event"]
)

# temporary → Validation / Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["pfi_event"]
)

print("Train      :", len(train_df),
      "| events:", int(train_df["pfi_event"].sum()))

print("Validation :", len(val_df),
      "| events:", int(val_df["pfi_event"].sum()))

print("Test       :", len(test_df),
      "| events:", int(test_df["pfi_event"].sum()))

print("\nOverlap")
print("Train-Val :", len(set(train_df.patient_id) & set(val_df.patient_id)))
print("Train-Test:", len(set(train_df.patient_id) & set(test_df.patient_id)))
print("Val-Test  :", len(set(val_df.patient_id) & set(test_df.patient_id)))

Train      : 370 | events: 111
Validation : 79 | events: 24
Test       : 80 | events: 24

Overlap
Train-Val : 0
Train-Test: 0
Val-Test  : 0


In [33]:
# ============================================================
# Reconstruct final Clinical + RNA preprocessing
# ============================================================

clinical_features = ["age", "gender", "stage", "grade"]

FINAL_COMBINED_K = 500
FINAL_COMBINED_L1 = 0.50
FINAL_COMBINED_ALPHA_IDX = 18


# ------------------------------------------------------------
# 1. Clinical preprocessing - fit on Train only
# ------------------------------------------------------------

final_combined_clin_preprocessor = ColumnTransformer(
    transformers=[
        ("age", "passthrough", ["age"]),
        (
            "cat",
            OneHotEncoder(
                categories=[
                    ["MALE", "FEMALE"],
                    ["Stage I", "Stage II", "Stage III", "Stage IV", "Unknown"],
                    ["G1", "G2", "G3", "G4", "Unknown"]
                ],
                drop=["MALE", "Stage I", "G2"],
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["gender", "stage", "grade"]
        )
    ]
)

X_train_clin = final_combined_clin_preprocessor.fit_transform(
    train_df[clinical_features]
)

X_val_clin = final_combined_clin_preprocessor.transform(
    val_df[clinical_features]
)

X_test_clin = final_combined_clin_preprocessor.transform(
    test_df[clinical_features]
)


# ------------------------------------------------------------
# 2. RNA preprocessing - determined from Train only
# ------------------------------------------------------------

rna_train_raw = rna_tpm.loc[train_df["patient_id"]]

min_samples = int(np.ceil(len(train_df) * 0.20))

keep_genes = (
    (rna_train_raw >= 1).sum(axis=0)
    >= min_samples
)

rna_train_filtered = rna_train_raw.loc[:, keep_genes]

rna_train_log = np.log2(rna_train_filtered + 1)

rna_train_median = rna_train_log.median(axis=0)

rna_train_mad = (
    rna_train_log
    .sub(rna_train_median, axis=1)
    .abs()
    .median(axis=0)
    .sort_values(ascending=False)
)

final_combined_genes = (
    rna_train_mad
    .head(FINAL_COMBINED_K)
    .index
    .tolist()
)


# Apply frozen Train-selected genes
X_train_rna = np.log2(
    rna_tpm.loc[
        train_df["patient_id"],
        final_combined_genes
    ] + 1
).to_numpy()

X_val_rna = np.log2(
    rna_tpm.loc[
        val_df["patient_id"],
        final_combined_genes
    ] + 1
).to_numpy()

X_test_rna = np.log2(
    rna_tpm.loc[
        test_df["patient_id"],
        final_combined_genes
    ] + 1
).to_numpy()


# ------------------------------------------------------------
# 3. Clinical + RNA
# ------------------------------------------------------------

X_train_combined_raw = np.hstack([
    X_train_clin,
    X_train_rna
])

X_val_combined_raw = np.hstack([
    X_val_clin,
    X_val_rna
])

X_test_combined_raw = np.hstack([
    X_test_clin,
    X_test_rna
])


# ------------------------------------------------------------
# 4. Scaling - fit on Train only
# ------------------------------------------------------------

final_combined_scaler = StandardScaler()

X_train_combined = final_combined_scaler.fit_transform(
    X_train_combined_raw
)

X_val_combined = final_combined_scaler.transform(
    X_val_combined_raw
)

X_test_combined = final_combined_scaler.transform(
    X_test_combined_raw
)


# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

print("Selected RNA genes :", len(final_combined_genes))
print("Clinical features  :", X_train_clin.shape[1])
print("Combined features  :", X_train_combined.shape[1])

print("\nTrain :", X_train_combined.shape)
print("Val   :", X_val_combined.shape)
print("Test  :", X_test_combined.shape)

Selected RNA genes : 500
Clinical features  : 10
Combined features  : 510

Train : (370, 510)
Val   : (79, 510)
Test  : (80, 510)


In [34]:
from sksurv.metrics import concordance_index_censored

# ------------------------------------------------------------
# Survival targets
# ------------------------------------------------------------

y_train = Surv.from_arrays(
    event=train_df["pfi_event"].astype(bool).to_numpy(),
    time=train_df["pfi_time"].astype(float).to_numpy()
)

y_val = Surv.from_arrays(
    event=val_df["pfi_event"].astype(bool).to_numpy(),
    time=val_df["pfi_time"].astype(float).to_numpy()
)


# ------------------------------------------------------------
# Reconstruct final Clinical + RNA CoxNet
# ------------------------------------------------------------

combined_coxnet_final = CoxnetSurvivalAnalysis(
    l1_ratio=FINAL_COMBINED_L1,
    n_alphas=50,
    alpha_min_ratio=0.01,
    max_iter=100000
)

combined_coxnet_final.fit(
    X_train_combined,
    y_train
)

combined_final_alpha = (
    combined_coxnet_final.alphas_[FINAL_COMBINED_ALPHA_IDX]
)


# ------------------------------------------------------------
# Validation risk
# ------------------------------------------------------------

val_risk = combined_coxnet_final.predict(
    X_val_combined,
    alpha=combined_final_alpha
)

val_cindex = concordance_index_censored(
    val_df["pfi_event"].astype(bool).to_numpy(),
    val_df["pfi_time"].astype(float).to_numpy(),
    val_risk
)[0]


print("Final alpha :", combined_final_alpha)
print("Val C-index :", val_cindex)

print(
    "Non-zero coefficients:",
    np.count_nonzero(
        combined_coxnet_final.coef_[:, FINAL_COMBINED_ALPHA_IDX]
    )
)

Final alpha : 0.1079631601142658
Val C-index : 0.8135158254918734
Non-zero coefficients: 24


In [35]:
# ============================================================
# Test-set reproduction check
# ============================================================

test_risk = combined_coxnet_final.predict(
    X_test_combined,
    alpha=combined_final_alpha
)

test_cindex = concordance_index_censored(
    test_df["pfi_event"].astype(bool).to_numpy(),
    test_df["pfi_time"].astype(float).to_numpy(),
    test_risk
)[0]

print("Test C-index:", test_cindex)

print("\nFirst 5 test patients:")
for patient_id, risk in zip(
    test_df["patient_id"].head(),
    test_risk[:5]
):
    print(patient_id, "->", risk)

Test C-index: 0.8919558359621451

First 5 test patients:
TCGA-BP-5169 -> 0.8775333495883141
TCGA-BP-4353 -> -0.038865330852587435
TCGA-BP-5200 -> 0.010716147441005639
TCGA-B0-5121 -> -0.34104747330052054
TCGA-B4-5378 -> -0.09107463517623635


In [36]:
matches = list(project_root.rglob("test_predictions.csv"))

print("Found:", len(matches))
for p in matches:
    print(p)

Found: 0


In [37]:
# ============================================================
# Single-patient Clinical + RNA risk inference
# ============================================================

def predict_clinical_rna_risk(patient_id):

    # 1. Patient 확인
    patient = cohort.loc[
        cohort["patient_id"] == patient_id
    ].copy()

    if len(patient) != 1:
        raise ValueError(
            f"patient_id '{patient_id}'를 정확히 1명 찾을 수 없습니다."
        )

    # --------------------------------------------------------
    # 2. Clinical preprocessing
    # --------------------------------------------------------

    X_clin = final_combined_clin_preprocessor.transform(
        patient[clinical_features]
    )

    # --------------------------------------------------------
    # 3. RNA preprocessing
    #    Train에서 이미 선택된 500 genes만 사용
    # --------------------------------------------------------

    X_rna = np.log2(
        rna_tpm.loc[
            [patient_id],
            final_combined_genes
        ] + 1
    ).to_numpy()

    # --------------------------------------------------------
    # 4. Clinical + RNA 결합
    # --------------------------------------------------------

    X_combined_raw = np.hstack([
        X_clin,
        X_rna
    ])

    # Train에서 학습된 scaler만 사용
    X_combined = final_combined_scaler.transform(
        X_combined_raw
    )

    # --------------------------------------------------------
    # 5. Frozen CoxNet risk score
    # --------------------------------------------------------

    risk_score = combined_coxnet_final.predict(
        X_combined,
        alpha=combined_final_alpha
    )[0]

    return float(risk_score)

In [38]:
patient_id = "TCGA-BP-5169"

single_risk = predict_clinical_rna_risk(patient_id)

print("Patient :", patient_id)
print("Risk    :", single_risk)
print("Expected:", 0.8775333495883141)
print("Match   :", np.isclose(
    single_risk,
    0.8775333495883141,
    rtol=1e-10,
    atol=1e-12
))

Patient : TCGA-BP-5169
Risk    : 0.8775333495883141
Expected: 0.8775333495883141
Match   : True


In [39]:
# 04_xai_interpretation.ipynb 불러오기
import json

xai_notebook_path = project_root / "notebooks" / "04_xai_interpretation.ipynb"

with open(xai_notebook_path, "r", encoding="utf-8") as f:
    nb04 = json.load(f)

# Local XAI 관련 셀 번호 찾기
for i, cell in enumerate(nb04["cells"]):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    keywords = [
        "contribution",
        "local",
        "risk_score"
    ]

    if any(k.lower() in src.lower() for k in keywords):
        first_line = next(
            (line.strip() for line in src.splitlines() if line.strip()),
            ""
        )
        print(f"CELL {i}: {first_line[:150]}")

CELL 15: # ============================================================
CELL 16: # ============================================================
CELL 17: # ============================================================
CELL 18: # ============================================================
CELL 20: # ============================================================
CELL 21: # ============================================================
CELL 22: # ============================================================


In [40]:
for i in range(15, 23):
    cell = nb04["cells"][i]

    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    lines = [line.strip() for line in src.splitlines() if line.strip()]

    print(f"\n=== CELL {i} ===")
    for line in lines[:6]:
        print(line[:180])


=== CELL 15 ===
# ============================================================
# Patient-specific feature contributions
# contribution_ij = standardized_value_ij × coefficient_j
# ============================================================
# Frozen coefficient
beta = recomputed_coef.copy()

=== CELL 16 ===
# ============================================================
# Check whether feature contributions reproduce model risk
# ============================================================
contribution_sum = (
contribution_df
.sum(axis=1)

=== CELL 17 ===
# ============================================================
# Final local contribution reconstruction check
# ============================================================
contribution_sum = (
contribution_df
.sum(axis=1)

=== CELL 18 ===
# ============================================================
# Keep only non-zero coefficient features
# ============================================================
nonzero_mask = np.abs(beta) >

In [41]:
cell15 = "".join(nb04["cells"][15]["source"])

for i, line in enumerate(cell15.splitlines(), start=1):
    print(f"{i:03d}: {line}")

001: # ============================================================
002: # Patient-specific feature contributions
003: # contribution_ij = standardized_value_ij × coefficient_j
004: # ============================================================
005: 
006: # Frozen coefficient
007: beta = recomputed_coef.copy()
008: 
009: # Test contribution matrix
010: test_contributions = (
011:     X_test_scaled * beta
012: )
013: 
014: # DataFrame
015: contribution_df = pd.DataFrame(
016:     test_contributions,
017:     index=test_df["patient_id"],
018:     columns=feature_names
019: )
020: 
021: print("Contribution matrix:", contribution_df.shape)
022: 
023: assert contribution_df.shape == (80, 510)


In [42]:
# ============================================================
# Single-patient Local XAI
# ============================================================

# Final CoxNet coefficients
beta = combined_coxnet_final.coef_[
    :, FINAL_COMBINED_ALPHA_IDX
]

# Feature names: Clinical 10 + RNA 500
clinical_feature_names = (
    final_combined_clin_preprocessor
    .get_feature_names_out()
    .tolist()
)

feature_names = (
    clinical_feature_names
    + final_combined_genes
)

assert len(feature_names) == 510
assert len(beta) == 510


def explain_patient(patient_id, top_n=10):

    # --------------------------------------------------------
    # 1. Patient
    # --------------------------------------------------------
    patient = cohort.loc[
        cohort["patient_id"] == patient_id
    ].copy()

    if len(patient) != 1:
        raise ValueError(
            f"patient_id '{patient_id}'를 정확히 1명 찾을 수 없습니다."
        )

    # --------------------------------------------------------
    # 2. Clinical
    # --------------------------------------------------------
    X_clin = final_combined_clin_preprocessor.transform(
        patient[clinical_features]
    )

    # --------------------------------------------------------
    # 3. RNA
    # --------------------------------------------------------
    X_rna = np.log2(
        rna_tpm.loc[
            [patient_id],
            final_combined_genes
        ] + 1
    ).to_numpy()

    # --------------------------------------------------------
    # 4. Combine + frozen scaling
    # --------------------------------------------------------
    X_raw = np.hstack([
        X_clin,
        X_rna
    ])

    X_scaled = final_combined_scaler.transform(
        X_raw
    )

    # --------------------------------------------------------
    # 5. Local contributions
    # --------------------------------------------------------
    contributions = X_scaled[0] * beta

    contribution_table = pd.DataFrame({
        "feature": feature_names,
        "standardized_value": X_scaled[0],
        "coefficient": beta,
        "contribution": contributions
    })

    contribution_table["abs_contribution"] = (
        contribution_table["contribution"].abs()
    )

    contribution_table = (
        contribution_table
        .sort_values(
            "abs_contribution",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 6. Risk reconstruction check
    # --------------------------------------------------------
    model_risk = predict_clinical_rna_risk(
        patient_id
    )

    contribution_sum = contributions.sum()

    print("Patient          :", patient_id)
    print("Model risk       :", model_risk)
    print("Contribution sum :", contribution_sum)
    print(
        "Match            :",
        np.isclose(
            model_risk,
            contribution_sum,
            rtol=1e-10,
            atol=1e-12
        )
    )

    return contribution_table.head(top_n)

In [43]:
xai_result = explain_patient(
    "TCGA-BP-5169",
    top_n=10
)

xai_result

Patient          : TCGA-BP-5169
Model risk       : 0.8775333495883141
Contribution sum : 0.8775333495883142
Match            : True


,feature,standardized_value,coefficient,contribution,abs_contribution
0,cat__grade_G4,2.529822,0.144119,0.364596,0.364596
1,cat__stage_Stage IV,-0.408892,0.615046,-0.251487,0.251487
2,ENSG00000147003.7,-1.693464,-0.102028,0.172781,0.172781
3,ENSG00000184434.8,-1.782942,-0.074754,0.133283,0.133283
4,ENSG00000074276.10,-1.431279,-0.063305,0.090608,0.090608
5,ENSG00000119411.11,-1.555822,-0.053348,0.082999,0.082999
6,ENSG00000164309.15,-1.477501,-0.039674,0.058619,0.058619
7,cat__stage_Stage III,-0.571108,0.102483,-0.058529,0.058529
8,ENSG00000103460.17,-1.149562,-0.045376,0.052163,0.052163
9,ENSG00000125255.7,-1.472003,-0.033247,0.048940,0.048940


In [44]:
# ============================================================
# Load Molecular Twin results
# ============================================================

twin_results = pd.read_csv(
    results_dir / "molecular_twin_results.csv"
)

print("Shape:", twin_results.shape)
print("Columns:", twin_results.columns.tolist())

twin_results.head()

Shape: (400, 9)
Columns: ['query_patient_id', 'twin_patient_id', 'similarity', 'age', 'gender', 'stage', 'grade', 'pfi_event', 'pfi_time']


,query_patient_id,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-BP-5169,TCGA-BP-4799,0.654253,70.0,MALE,Stage III,G3,1.0,352.0
1,TCGA-BP-5169,TCGA-BP-4352,0.585815,74.0,FEMALE,Stage IV,G4,1.0,13.0
2,TCGA-BP-5169,TCGA-B8-A54I,0.585734,48.0,MALE,Stage I,G3,0.0,150.0
3,TCGA-BP-5169,TCGA-G6-A8L7,0.582847,81.0,FEMALE,Stage I,G3,0.0,2133.0
4,TCGA-BP-5169,TCGA-BP-4761,0.568247,57.0,MALE,Stage III,G4,0.0,182.0


In [45]:
# ============================================================
# Molecular Twin lookup
# ============================================================

def get_molecular_twins(patient_id, top_n=5):

    twins = (
        twin_results[
            twin_results["query_patient_id"] == patient_id
        ]
        .sort_values("similarity", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
        .copy()
    )

    if len(twins) == 0:
        raise ValueError(
            f"'{patient_id}'의 Molecular Twin 결과가 없습니다."
        )

    return twins


# Test
patient_id = "TCGA-BP-5169"

patient_twins = get_molecular_twins(
    patient_id,
    top_n=5
)

patient_twins

,query_patient_id,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-BP-5169,TCGA-BP-4799,0.654253,70.0,MALE,Stage III,G3,1.0,352.0
1,TCGA-BP-5169,TCGA-BP-4352,0.585815,74.0,FEMALE,Stage IV,G4,1.0,13.0
2,TCGA-BP-5169,TCGA-B8-A54I,0.585734,48.0,MALE,Stage I,G3,0.0,150.0
3,TCGA-BP-5169,TCGA-G6-A8L7,0.582847,81.0,FEMALE,Stage I,G3,0.0,2133.0
4,TCGA-BP-5169,TCGA-BP-4761,0.568247,57.0,MALE,Stage III,G4,0.0,182.0


In [46]:
# ============================================================
# Load Molecular Support results
# ============================================================

support_results = pd.read_csv(
    results_dir / "patient_molecular_support_summary.csv"
)

print("Shape:", support_results.shape)
print("Columns:", support_results.columns.tolist())

support_results.head()

Shape: (80, 11)
Columns: ['patient_id', 'age', 'gender', 'stage', 'grade', 'nearest_train_patient', 'max_similarity', 'ood_threshold', 'low_support', 'pfi_event', 'pfi_time']


,patient_id,age,gender,stage,grade,nearest_train_patient,max_similarity,ood_threshold,low_support,pfi_event,pfi_time
0,TCGA-BP-5169,70.0,MALE,Stage I,G4,TCGA-BP-4799,0.654253,0.380982,False,0.0,193.0
1,TCGA-BP-4353,61.0,MALE,Stage I,G2,TCGA-A3-3323,0.489591,0.380982,False,0.0,375.0
2,TCGA-BP-5200,44.0,MALE,Stage II,G4,TCGA-CJ-4904,0.460775,0.380982,False,0.0,1063.0
3,TCGA-B0-5121,56.0,MALE,Stage I,G2,TCGA-B8-4153,0.533545,0.380982,False,0.0,1485.0
4,TCGA-B4-5378,62.0,MALE,Stage I,G2,TCGA-DV-5567,0.846025,0.380982,False,0.0,175.0


In [47]:
# ============================================================
# Molecular Support lookup
# ============================================================

def get_molecular_support(patient_id):

    support = support_results.loc[
        support_results["patient_id"] == patient_id
    ].copy()

    if len(support) != 1:
        raise ValueError(
            f"'{patient_id}'의 Molecular Support 결과를 "
            "정확히 1개 찾을 수 없습니다."
        )

    row = support.iloc[0]

    return {
        "nearest_train_patient": row["nearest_train_patient"],
        "max_similarity": float(row["max_similarity"]),
        "support_threshold": float(row["ood_threshold"]),
        "low_support": bool(row["low_support"])
    }


# Test
get_molecular_support("TCGA-BP-5169")

{'nearest_train_patient': 'TCGA-BP-4799',
 'max_similarity': 0.65425265,
 'support_threshold': 0.38098168,
 'low_support': False}

In [48]:
# ============================================================
# Unified Patient-level Inference
# ============================================================

def predict_patient(patient_id, top_xai=10, top_twins=5):

    # --------------------------------------------------------
    # 1. Patient information
    # --------------------------------------------------------
    patient = cohort.loc[
        cohort["patient_id"] == patient_id
    ].copy()

    if len(patient) != 1:
        raise ValueError(
            f"patient_id '{patient_id}'를 정확히 1명 찾을 수 없습니다."
        )

    patient_info = patient.iloc[0]

    # --------------------------------------------------------
    # 2. Clinical + RNA CoxNet risk
    # --------------------------------------------------------
    risk_score = predict_clinical_rna_risk(
        patient_id
    )

    # --------------------------------------------------------
    # 3. Local XAI
    # --------------------------------------------------------
    xai = explain_patient(
        patient_id,
        top_n=top_xai
    )

    # --------------------------------------------------------
    # 4. Molecular Twins
    # --------------------------------------------------------
    twins = get_molecular_twins(
        patient_id,
        top_n=top_twins
    )

    # --------------------------------------------------------
    # 5. Molecular Support
    # --------------------------------------------------------
    support = get_molecular_support(
        patient_id
    )

    # --------------------------------------------------------
    # 6. Unified result
    # --------------------------------------------------------
    result = {
        "patient_id": patient_id,

        "clinical": {
            "age": float(patient_info["age"]),
            "gender": patient_info["gender"],
            "stage": patient_info["stage"],
            "grade": patient_info["grade"]
        },

        "risk_score": risk_score,

        "molecular_support": support,

        "top_contributions": xai,

        "molecular_twins": twins
    }

    return result

In [49]:
result = predict_patient(
    "TCGA-BP-5169"
)

print("\n=== PATIENT ===")
print(result["patient_id"])
print(result["clinical"])

print("\n=== RISK SCORE ===")
print(result["risk_score"])

print("\n=== MOLECULAR SUPPORT ===")
print(result["molecular_support"])

print("\n=== TOP XAI ===")
display(result["top_contributions"])

print("\n=== MOLECULAR TWINS ===")
display(result["molecular_twins"])

Patient          : TCGA-BP-5169
Model risk       : 0.8775333495883141
Contribution sum : 0.8775333495883142
Match            : True

=== PATIENT ===
TCGA-BP-5169
{'age': 70.0, 'gender': 'MALE', 'stage': 'Stage I', 'grade': 'G4'}

=== RISK SCORE ===
0.8775333495883141

=== MOLECULAR SUPPORT ===
{'nearest_train_patient': 'TCGA-BP-4799', 'max_similarity': 0.65425265, 'support_threshold': 0.38098168, 'low_support': False}

=== TOP XAI ===


,feature,standardized_value,coefficient,contribution,abs_contribution
0,cat__grade_G4,2.529822,0.144119,0.364596,0.364596
1,cat__stage_Stage IV,-0.408892,0.615046,-0.251487,0.251487
2,ENSG00000147003.7,-1.693464,-0.102028,0.172781,0.172781
3,ENSG00000184434.8,-1.782942,-0.074754,0.133283,0.133283
4,ENSG00000074276.10,-1.431279,-0.063305,0.090608,0.090608
5,ENSG00000119411.11,-1.555822,-0.053348,0.082999,0.082999
6,ENSG00000164309.15,-1.477501,-0.039674,0.058619,0.058619
7,cat__stage_Stage III,-0.571108,0.102483,-0.058529,0.058529
8,ENSG00000103460.17,-1.149562,-0.045376,0.052163,0.052163
9,ENSG00000125255.7,-1.472003,-0.033247,0.048940,0.048940



=== MOLECULAR TWINS ===


,query_patient_id,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-BP-5169,TCGA-BP-4799,0.654253,70.0,MALE,Stage III,G3,1.0,352.0
1,TCGA-BP-5169,TCGA-BP-4352,0.585815,74.0,FEMALE,Stage IV,G4,1.0,13.0
2,TCGA-BP-5169,TCGA-B8-A54I,0.585734,48.0,MALE,Stage I,G3,0.0,150.0
3,TCGA-BP-5169,TCGA-G6-A8L7,0.582847,81.0,FEMALE,Stage I,G3,0.0,2133.0
4,TCGA-BP-5169,TCGA-BP-4761,0.568247,57.0,MALE,Stage III,G4,0.0,182.0


In [50]:
# ============================================================
# Load gene annotation for Patient Report
# ============================================================

gene_annotation = pd.read_parquet(
    data_dir / "interim" / "rna_gene_annotation.parquet"
)

print("Shape:", gene_annotation.shape)
print("Columns:", gene_annotation.columns.tolist())

gene_annotation.head()

Shape: (19962, 3)
Columns: ['gene_id', 'gene_name', 'gene_type']


,gene_id,gene_name,gene_type
0,ENSG00000000003.15,TSPAN6,protein_coding
1,ENSG00000000005.6,TNMD,protein_coding
2,ENSG00000000419.13,DPM1,protein_coding
3,ENSG00000000457.14,SCYL3,protein_coding
4,ENSG00000000460.17,C1orf112,protein_coding


In [51]:
# ============================================================
# Patient Report
# ============================================================

# Ensembl ID -> Gene symbol
gene_name_map = (
    gene_annotation
    .set_index("gene_id")["gene_name"]
    .to_dict()
)


def make_patient_report(patient_id, top_xai=10, top_twins=5):

    # 기존 통합 inference 실행
    result = predict_patient(
        patient_id,
        top_xai=top_xai,
        top_twins=top_twins
    )

    # --------------------------------------------------------
    # 1. XAI table 정리
    # --------------------------------------------------------
    xai = result["top_contributions"].copy()

    def make_feature_label(feature):

        # Clinical feature
        if feature.startswith("age__") or feature.startswith("cat__"):
            return feature.replace("age__", "").replace("cat__", "")

        # RNA feature
        gene_name = gene_name_map.get(feature)

        if gene_name is not None:
            return f"{gene_name} ({feature})"

        return feature

    xai["feature_label"] = xai["feature"].apply(
        make_feature_label
    )

    xai["direction"] = np.where(
        xai["contribution"] > 0,
        "Higher model risk",
        "Lower model risk"
    )

    xai_report = xai[
        [
            "feature_label",
            "contribution",
            "direction"
        ]
    ].copy()

    # --------------------------------------------------------
    # 2. Twin table 정리
    # --------------------------------------------------------
    twins = result["molecular_twins"].copy()

    twin_report = twins[
        [
            "twin_patient_id",
            "similarity",
            "age",
            "gender",
            "stage",
            "grade",
            "pfi_event",
            "pfi_time"
        ]
    ].copy()

    # --------------------------------------------------------
    # 3. Report 출력
    # --------------------------------------------------------
    clinical = result["clinical"]
    support = result["molecular_support"]

    print("=" * 65)
    print("TCGA-KIRC PATIENT REPORT")
    print("=" * 65)

    print(f"\nPatient ID : {patient_id}")

    print("\n[Clinical]")
    print(f"Age        : {clinical['age']:.0f}")
    print(f"Gender     : {clinical['gender']}")
    print(f"Stage      : {clinical['stage']}")
    print(f"Grade      : {clinical['grade']}")

    print("\n[Clinical + RNA CoxNet]")
    print(
        f"Risk score : {result['risk_score']:.4f}"
    )
    print(
        "(Relative Cox model score; "
        "not an absolute recurrence probability)"
    )

    print("\n[Molecular Support]")
    print(
        f"Nearest Train patient : "
        f"{support['nearest_train_patient']}"
    )
    print(
        f"Maximum similarity    : "
        f"{support['max_similarity']:.4f}"
    )
    print(
        f"Support threshold     : "
        f"{support['support_threshold']:.4f}"
    )
    print(
        f"Low molecular support : "
        f"{support['low_support']}"
    )

    print("\n[Top Local Contributions]")
    display(xai_report)

    print("\n[Top Molecular Twins]")
    display(twin_report)

    return {
        "patient_id": patient_id,
        "clinical": clinical,
        "risk_score": result["risk_score"],
        "molecular_support": support,
        "xai": xai_report,
        "molecular_twins": twin_report
    }

In [52]:
patient_report = make_patient_report(
    "TCGA-BP-5169"
)

Patient          : TCGA-BP-5169
Model risk       : 0.8775333495883141
Contribution sum : 0.8775333495883142
Match            : True
TCGA-KIRC PATIENT REPORT

Patient ID : TCGA-BP-5169

[Clinical]
Age        : 70
Gender     : MALE
Stage      : Stage I
Grade      : G4

[Clinical + RNA CoxNet]
Risk score : 0.8775
(Relative Cox model score; not an absolute recurrence probability)

[Molecular Support]
Nearest Train patient : TCGA-BP-4799
Maximum similarity    : 0.6543
Support threshold     : 0.3810
Low molecular support : False

[Top Local Contributions]


,feature_label,contribution,direction
0,grade_G4,0.364596,Higher model risk
1,stage_Stage IV,-0.251487,Lower model risk
2,CLTRN (ENSG00000147003.7),0.172781,Higher model risk
3,LRRC19 (ENSG00000184434.8),0.133283,Higher model risk
4,CDHR2 (ENSG00000074276.10),0.090608,Higher model risk
5,BSPRY (ENSG00000119411.11),0.082999,Higher model risk
6,CMYA5 (ENSG00000164309.15),0.058619,Higher model risk
7,stage_Stage III,-0.058529,Lower model risk
8,TOX3 (ENSG00000103460.17),0.052163,Higher model risk
9,SLC10A2 (ENSG00000125255.7),0.048940,Higher model risk



[Top Molecular Twins]


,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-BP-4799,0.654253,70.0,MALE,Stage III,G3,1.0,352.0
1,TCGA-BP-4352,0.585815,74.0,FEMALE,Stage IV,G4,1.0,13.0
2,TCGA-B8-A54I,0.585734,48.0,MALE,Stage I,G3,0.0,150.0
3,TCGA-G6-A8L7,0.582847,81.0,FEMALE,Stage I,G3,0.0,2133.0
4,TCGA-BP-4761,0.568247,57.0,MALE,Stage III,G4,0.0,182.0


In [53]:
# ============================================================
# Full Test-set QC
# ============================================================

qc_rows = []

beta = combined_coxnet_final.coef_[
    :, FINAL_COMBINED_ALPHA_IDX
]

for i, patient_id in enumerate(test_df["patient_id"]):

    # ----------------------------------------
    # 1. Single-patient inference
    # ----------------------------------------
    single_risk = predict_clinical_rna_risk(
        patient_id
    )

    # 기존 batch prediction
    batch_risk = float(test_risk[i])

    # ----------------------------------------
    # 2. Reconstruct patient input
    # ----------------------------------------
    patient = cohort.loc[
        cohort["patient_id"] == patient_id
    ]

    X_clin = final_combined_clin_preprocessor.transform(
        patient[clinical_features]
    )

    X_rna = np.log2(
        rna_tpm.loc[
            [patient_id],
            final_combined_genes
        ] + 1
    ).to_numpy()

    X_raw = np.hstack([
        X_clin,
        X_rna
    ])

    X_scaled = final_combined_scaler.transform(
        X_raw
    )

    # ----------------------------------------
    # 3. Local XAI reconstruction
    # ----------------------------------------
    contribution_sum = float(
        np.sum(X_scaled[0] * beta)
    )

    qc_rows.append({
        "patient_id": patient_id,
        "batch_risk": batch_risk,
        "single_risk": single_risk,
        "contribution_sum": contribution_sum,
        "risk_diff": abs(batch_risk - single_risk),
        "xai_diff": abs(single_risk - contribution_sum)
    })


qc_df = pd.DataFrame(qc_rows)

print("Patients checked :", len(qc_df))

print(
    "Max batch-single difference:",
    qc_df["risk_diff"].max()
)

print(
    "Max risk-XAI difference:",
    qc_df["xai_diff"].max()
)

print(
    "All risk matches:",
    np.allclose(
        qc_df["batch_risk"],
        qc_df["single_risk"],
        rtol=1e-10,
        atol=1e-12
    )
)

print(
    "All XAI matches:",
    np.allclose(
        qc_df["single_risk"],
        qc_df["contribution_sum"],
        rtol=1e-10,
        atol=1e-12
    )
)

qc_df.head()

Patients checked : 80
Max batch-single difference: 4.440892098500626e-16
Max risk-XAI difference: 4.440892098500626e-16
All risk matches: True
All XAI matches: True


,patient_id,batch_risk,single_risk,contribution_sum,risk_diff,xai_diff
0,TCGA-BP-5169,0.877533,0.877533,0.877533,0.000000e+00,1.110223e-16
1,TCGA-BP-4353,-0.038865,-0.038865,-0.038865,1.110223e-16,1.526557e-16
2,TCGA-BP-5200,0.010716,0.010716,0.010716,2.428613e-17,7.112366e-17
3,TCGA-B0-5121,-0.341047,-0.341047,-0.341047,0.000000e+00,1.110223e-16
4,TCGA-B4-5378,-0.091075,-0.091075,-0.091075,4.163336e-17,1.387779e-17


In [54]:
# ============================================================
# Save Patient Inference QC
# ============================================================

qc_output_path = (
    results_dir / "patient_inference_qc.csv"
)

qc_df.to_csv(
    qc_output_path,
    index=False
)

print("Saved:", qc_output_path)
print("Rows :", len(qc_df))

Saved: c:\kirc-genomic-prognosis\results\patient_inference_qc.csv
Rows : 80
